In [0]:
%sql
MERGE INTO ecomm.gold.dim_customer d
USING ecomm.silver.customers_current s
   ON d.customer_unique_id = s.customer_unique_id
  AND d.is_current
WHEN MATCHED AND (d.customer_city  <> s.customer_city
               OR d.customer_state <> s.customer_state
               OR d.customer_zip_prefix <> s.customer_zip_prefix)
THEN UPDATE SET d.valid_to = current_timestamp(), d.is_current = false;

INSERT INTO ecomm.gold.dim_customer
SELECT
    md5(concat(s.customer_unique_id, '|', cast(current_timestamp() AS STRING))),
    s.customer_unique_id, s.customer_city, s.customer_state, s.customer_zip_prefix,
    current_timestamp(), NULL, true
FROM ecomm.silver.customers_current s
LEFT JOIN ecomm.gold.dim_customer d
       ON d.customer_unique_id = s.customer_unique_id AND d.is_current
WHERE d.customer_unique_id IS NULL
  AND NOT s._is_deleted;